# 04 Feature Engineering

Notebook ini membentuk fitur transaksi yang dipakai untuk deteksi impulsive spending. Semua proses dibuat display-only dan reusable agar bisa dijalankan dari hasil cleaning di memory atau dari CSV cleaned read-only.

Output utama di memory:
- `df_features`: dataframe transaksi dengan fitur baru.
- `feature_summary_df`: ringkasan fitur per dataset.
- `new_feature_columns`: daftar kolom fitur yang ditambahkan.


## 1. Import Library


In [1]:
from pathlib import Path
import warnings

import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display


## 2. Konfigurasi Path dan Tampilan


In [2]:
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

cleaned_separate_path = project_root / 'data' / 'interim' / 'cleaned_separate'

alt.data_transformers.disable_max_rows()
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
warnings.filterwarnings('ignore')

print(f'Project root         : {project_root}')
print(f'Cleaned separate path: {cleaned_separate_path}')
print('Mode                 : display-only. Tidak ada file output yang disimpan.')


Project root         : /home/umaygans/05_nayyara_submission_1/nayyara_capstone
Cleaned separate path: /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/interim/cleaned_separate
Mode                 : display-only. Tidak ada file output yang disimpan.


## 3. Bobot Fingo Signal


In [3]:
W_NIGHT = 0.35
W_CATEGORY = 0.30
W_AMOUNT = 0.25
W_WEEKEND = 0.10


## 4. Konfigurasi Kategori


In [4]:
STANDARD_CATEGORIES = ['Makanan', 'Transportasi', 'Hiburan', 'Belanja', 'Pendidikan', 'Kesehatan', 'Tagihan', 'Lainnya']
HEDONIC_CATEGORIES = {'Hiburan', 'Belanja'}
NEUTRAL_CATEGORIES = {'Makanan', 'Transportasi', 'Lainnya'}
UTIL_CATEGORIES = {'Pendidikan', 'Kesehatan', 'Tagihan'}

DIRECT_CATEGORY_MAP = {
    'food': 'Makanan',
    'food_and_drink': 'Makanan',
    'food_drink': 'Makanan',
    'grocery': 'Makanan',
    'groceries': 'Makanan',
    'restaurant': 'Makanan',
    'snacks': 'Makanan',
    'makanan': 'Makanan',
    'transportation': 'Transportasi',
    'transport': 'Transportasi',
    'travel': 'Transportasi',
    'commute': 'Transportasi',
    'train': 'Transportasi',
    'bus': 'Transportasi',
    'transportasi': 'Transportasi',
    'entertainment': 'Hiburan',
    'subscription': 'Hiburan',
    'movies': 'Hiburan',
    'gaming': 'Hiburan',
    'culture': 'Hiburan',
    'festivals': 'Hiburan',
    'tourism': 'Hiburan',
    'social_life': 'Hiburan',
    'hiburan': 'Hiburan',
    'shopping': 'Belanja',
    'apparel': 'Belanja',
    'clothing': 'Belanja',
    'beauty': 'Belanja',
    'grooming': 'Belanja',
    'household': 'Belanja',
    'gift': 'Belanja',
    'belanja': 'Belanja',
    'education': 'Pendidikan',
    'self_development': 'Pendidikan',
    'pendidikan': 'Pendidikan',
    'health': 'Kesehatan',
    'health_and_fitness': 'Kesehatan',
    'fitness': 'Kesehatan',
    'medical': 'Kesehatan',
    'kesehatan': 'Kesehatan',
    'utilities': 'Tagihan',
    'rent': 'Tagihan',
    'bills': 'Tagihan',
    'mobile_service_provider': 'Tagihan',
    'water': 'Tagihan',
    'internet': 'Tagihan',
    'electricity': 'Tagihan',
    'phone': 'Tagihan',
    'tagihan': 'Tagihan',
    'other': 'Lainnya',
    'unknown': 'Lainnya',
    'lainnya': 'Lainnya',
}

KEYWORD_CATEGORY_RULES = [
    ('Makanan', ['makanan', 'minuman', 'food', 'snack', 'grocery', 'restaurant']),
    ('Transportasi', ['transport', 'commute', 'travel', 'train', 'bus', 'ojek']),
    ('Hiburan', ['entertainment', 'subscription', 'movie', 'gaming', 'mainan', 'festival', 'netflix', 'culture']),
    (
        'Belanja',
        [
            'shopping', 'fashion', 'pakaian', 'apparel', 'beauty', 'aksesoris', 'plastik', 'wadah', 'rak',
            'celengan', 'nampan', 'tray', 'baskom', 'mangkok', 'lunch_box', 'rantang', 'saringan',
            'pintu', 'perkakas', 'seal', 'baut', 'roof', 'household',
        ],
    ),
    ('Pendidikan', ['education', 'school', 'course', 'book', 'buku']),
    ('Kesehatan', ['health', 'fitness', 'medical', 'obat', 'olahraga']),
    ('Tagihan', ['utilities', 'rent', 'bill', 'mobile', 'water', 'internet', 'electricity', 'phone', 'listrik', 'pulsa']),
]


## 5. Category Helper


In [5]:
def clean_key(value):
    text = str(value).strip().lower().replace('&', ' and ')
    text = ''.join(character if character.isalnum() else '_' for character in text)

    while '__' in text:
        text = text.replace('__', '_')

    return text.strip('_')


def map_category(value, domain=None):
    key = clean_key(value)

    if key in DIRECT_CATEGORY_MAP:
        return DIRECT_CATEGORY_MAP[key]

    for category, keywords in KEYWORD_CATEGORY_RULES:
        if any(keyword in key for keyword in keywords):
            return category

    return 'Belanja' if domain == 'ecommerce_sales' else 'Lainnya'


## 6. Load Cleaned Data Helper


In [6]:
def load_cleaned_from_csv(folder):
    frames = []

    for path in sorted(folder.glob('*_cleaned.csv')):
        frame = pd.read_csv(path, low_memory=False)
        if 'timestamp' not in frame.columns and 'date' in frame.columns:
            frame['timestamp'] = frame['date']
        if 'dataset_id' not in frame.columns:
            frame['dataset_id'] = path.name.replace('_cleaned.csv', '')
        frames.append(frame)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def prepare_cleaned_input(frame):
    data = frame.copy()
    required_columns = ['timestamp', 'amount', 'category', 'dataset_id']
    missing_columns = [column for column in required_columns if column not in data.columns]

    if missing_columns:
        raise ValueError(f'Kolom cleaned wajib belum ada: {missing_columns}')

    data['timestamp'] = pd.to_datetime(data['timestamp'], errors='coerce')
    data['amount'] = pd.to_numeric(data['amount'], errors='coerce')
    data = data.dropna(subset=required_columns).query('amount > 0').copy()
    data['category'] = data.apply(lambda row: map_category(row['category'], row.get('domain')), axis=1)
    data['source'] = data['source'] if 'source' in data.columns else data['dataset_id']
    return data.sort_values('timestamp').reset_index(drop=True)


def resolve_cleaned_input(cleaned_frame=None):
    if isinstance(cleaned_frame, pd.DataFrame) and not cleaned_frame.empty:
        return prepare_cleaned_input(cleaned_frame), 'memory_from_03'

    return prepare_cleaned_input(load_cleaned_from_csv(cleaned_separate_path)), 'read_only_csv_fallback'


def build_input_summary(frame):
    return (
        frame.groupby('dataset_id')
        .agg(rows=('amount', 'size'), total_amount=('amount', 'sum'), median_amount=('amount', 'median'))
        .reset_index()
    )


## 7. Load Cleaned Data


In [7]:
df_input, input_mode = resolve_cleaned_input(globals().get('df_cleaned'))

if df_input.empty:
    raise FileNotFoundError('Tidak ada data cleaned. Jalankan 03 dulu atau sediakan cleaned CSV read-only.')

input_summary_df = build_input_summary(df_input)
print(f'Input mode: {input_mode}')
display(input_summary_df)
display(df_input.head())


Input mode: read_only_csv_fallback


,dataset_id,rows,total_amount,median_amount
0,daily_household_transactions,2445,5898107.98,100.0
1,ecommerce_2024_01_january,367,23760242.36,35400.0
2,ecommerce_2024_06_june,608,30399557.00,25248.0
3,ecommerce_2025_02_february,831,48015557.90,22900.0
4,ecommerce_2025_11_november,1022,40747696.90,19500.0


,dataset_id,dataset_name,domain,source_file,dataset_period,date,amount,amount_original,category,description,transaction_type,status,payment_method,city,province,quantity,order_id,date_source,amount_cap_p99,month,year,timestamp,source
0,daily_household_transactions,Daily Household Transactions,household_finance,Daily Household Transactions.csv,NaN,2015-01-01 00:00:00,10.0,10.0,Makanan,tea,Expense,unknown,Cash,unknown,unknown,NaN,unknown,source_column,62292.8,2015-01,2015,2015-01-01,daily_household_transactions
1,daily_household_transactions,Daily Household Transactions,household_finance,Daily Household Transactions.csv,NaN,2015-01-01 00:00:00,10.0,10.0,Transportasi,share auto - hospital to brc station,Expense,unknown,Cash,unknown,unknown,NaN,unknown,source_column,62292.8,2015-01,2015,2015-01-01,daily_household_transactions
2,daily_household_transactions,Daily Household Transactions,household_finance,Daily Household Transactions.csv,NaN,2015-01-01 00:00:00,400.0,400.0,Makanan,bendys chicken biryani,Expense,unknown,Credit Card,unknown,unknown,NaN,unknown,source_column,62292.8,2015-01,2015,2015-01-01,daily_household_transactions
3,daily_household_transactions,Daily Household Transactions,household_finance,Daily Household Transactions.csv,NaN,2015-01-01 00:00:00,20.0,20.0,Transportasi,share jeep - Place T top to base,Expense,unknown,Cash,unknown,unknown,NaN,unknown,source_column,62292.8,2015-01,2015,2015-01-01,daily_household_transactions
4,daily_household_transactions,Daily Household Transactions,household_finance,Daily Household Transactions.csv,NaN,2015-01-01 00:00:00,60.0,60.0,Transportasi,share jeep - Place T to brc,Expense,unknown,Cash,unknown,unknown,NaN,unknown,source_column,62292.8,2015-01,2015,2015-01-01,daily_household_transactions


## 8. Feature Helpers


In [8]:
def add_time_features(frame):
    data = frame.copy()
    data['hour'] = data['timestamp'].dt.hour
    data['day_of_week'] = data['timestamp'].dt.dayofweek
    data['day_name'] = data['timestamp'].dt.day_name()
    data['day_of_month'] = data['timestamp'].dt.day
    data['month'] = data['timestamp'].dt.to_period('M').astype(str)
    data['date_only'] = data['timestamp'].dt.date.astype(str)
    data['is_weekend'] = data['day_of_week'].isin([5, 6]).astype(int)
    data['is_night'] = ((data['hour'] >= 20) | (data['hour'] <= 3)).astype(int)
    data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
    data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)
    data['night_score'] = (1 - (np.minimum(data['hour'], 24 - data['hour']) / 12)).clip(0, 1)
    data['is_payday_window'] = (data['day_of_month'].between(25, 31) | data['day_of_month'].between(1, 5)).astype(int)
    data['time_segment'] = pd.cut(
        data['hour'],
        bins=[-1, 5, 10, 15, 19, 23],
        labels=['late_night', 'morning', 'midday', 'evening', 'night'],
    ).astype('string')
    return data


def add_category_features(frame):
    data = frame.copy()
    score_map = {category: 0.0 for category in STANDARD_CATEGORIES}
    score_map.update({category: 1.0 for category in HEDONIC_CATEGORIES})
    score_map.update({category: 0.5 for category in NEUTRAL_CATEGORIES})

    data['category_score'] = data['category'].map(score_map).fillna(0)
    data['category_type'] = np.select(
        [
            data['category'].isin(HEDONIC_CATEGORIES),
            data['category'].isin(NEUTRAL_CATEGORIES),
            data['category'].isin(UTIL_CATEGORIES),
        ],
        ['hedonic', 'neutral', 'utilitarian'],
        default='other',
    )
    data['is_hedonic_category'] = data['category'].isin(HEDONIC_CATEGORIES).astype(int)
    return data


def robust_z_score(series):
    values = pd.to_numeric(series, errors='coerce')
    median = values.median()
    mad = (values - median).abs().median()

    if pd.isna(mad) or mad == 0:
        std = values.std(ddof=0)
        if pd.isna(std) or std == 0:
            return pd.Series(0.0, index=series.index)
        return (values - values.mean()) / (std + 1e-9)

    return 0.6745 * (values - median) / (mad + 1e-9)


def add_amount_features(frame):
    data = frame.copy()
    data['amount_log'] = np.log1p(data['amount'])
    data['amount_z'] = data.groupby('dataset_id', group_keys=False)['amount'].transform(robust_z_score).clip(-3, 3)
    data['amount_score'] = ((data['amount_z'] + 3) / 6).clip(0, 1)
    data['amount_percentile'] = data.groupby('dataset_id')['amount'].rank(pct=True)
    data['is_high_amount'] = (data['amount_percentile'] >= 0.90).astype(int)
    return data


def add_context_features(frame):
    data = frame.copy().sort_values('timestamp').reset_index(drop=True)
    city = data.get('city', pd.Series('unknown', index=data.index)).astype('string').fillna('unknown')
    province = data.get('province', pd.Series('unknown', index=data.index)).astype('string').fillna('unknown')
    user_id = data.get('user_id', pd.Series('', index=data.index)).astype('string').fillna('')
    fallback_user = data['dataset_id'].astype(str) + '_' + city + '_' + province

    data['user_proxy'] = user_id.mask(user_id.str.strip().isin(['', '<NA>', 'nan']), fallback_user)
    data['transactions_same_day'] = data.groupby(['user_proxy', 'date_only'])['amount'].transform('size')
    data['daily_amount_total'] = data.groupby(['user_proxy', 'date_only'])['amount'].transform('sum')
    data['share_of_daily_spend'] = (data['amount'] / (data['daily_amount_total'] + 1e-9)).clip(0, 1)
    data['amount_vs_daily_avg'] = (
        data['amount'] / (data.groupby(['user_proxy', 'date_only'])['amount'].transform('mean') + 1e-9)
    ).clip(0, 5)
    return data


def add_fingo_signal(frame):
    data = frame.copy()
    data['weekend_score'] = data['is_weekend'].astype(float)
    data['fingo_impulse_signal'] = (
        W_NIGHT * data['night_score']
        + W_CATEGORY * data['category_score']
        + W_AMOUNT * data['amount_score']
        + W_WEEKEND * data['weekend_score']
    ).clip(0, 1)
    data['signal_band'] = pd.cut(
        data['fingo_impulse_signal'],
        bins=[-0.001, 0.35, 0.55, 0.75, 1.0],
        labels=['low', 'medium', 'watch', 'high'],
    ).astype('string')
    return data


## 9. Run Feature Pipeline Helper


In [9]:
def run_feature_engineering(frame):
    data = add_time_features(frame)
    data = add_category_features(data)
    data = add_amount_features(data)
    data = add_context_features(data)
    data = add_fingo_signal(data)
    return data


def get_new_feature_columns(input_frame, output_frame):
    base_columns = set(input_frame.columns)
    return [column for column in output_frame.columns if column not in base_columns]


## 10. Run Feature Pipeline


In [10]:
df_features = run_feature_engineering(df_input)
new_feature_columns = get_new_feature_columns(df_input, df_features)

print(f'Input shape : {df_input.shape}')
print(f'Output shape: {df_features.shape}')
print(f'Fitur baru  : {len(new_feature_columns)} kolom')
display(pd.DataFrame({'new_feature_columns': new_feature_columns}))
display(df_features.head())


Input shape : (5273, 23)
Output shape: (5273, 51)
Fitur baru  : 28 kolom


,new_feature_columns
0,hour
1,day_of_week
2,day_name
3,day_of_month
4,date_only
5,is_weekend
6,is_night
7,hour_sin
8,hour_cos
9,night_score


,dataset_id,dataset_name,domain,source_file,dataset_period,date,amount,amount_original,category,description,transaction_type,status,payment_method,city,province,quantity,order_id,date_source,amount_cap_p99,month,year,timestamp,source,hour,day_of_week,day_name,day_of_month,date_only,is_weekend,is_night,hour_sin,hour_cos,night_score,is_payday_window,time_segment,category_score,category_type,is_hedonic_category,amount_log,amount_z,amount_score,amount_percentile,is_high_amount,user_proxy,transactions_same_day,daily_amount_total,share_of_daily_spend,amount_vs_daily_avg,weekend_score,fingo_impulse_signal,signal_band
0,daily_household_transactions,Daily Household Transactions,household_finance,Daily Household Transactions.csv,NaN,2015-01-01 00:00:00,10.0,10.0,Makanan,tea,Expense,unknown,Cash,unknown,unknown,NaN,unknown,source_column,62292.8,2015-01,2015,2015-01-01,daily_household_transactions,0,3,Thursday,1,2015-01-01,0,1,0.0,1.0,1.0,1,late_night,0.5,neutral,0,2.397895,-0.714176,0.380971,0.037628,0,daily_household_transactions_unknown_unknown,11,952.0,0.010504,0.115546,0.0,0.595243,watch
1,daily_household_transactions,Daily Household Transactions,household_finance,Daily Household Transactions.csv,NaN,2015-01-01 00:00:00,142.0,142.0,Transportasi,ropeway Place T to and fro,Expense,unknown,Cash,unknown,unknown,NaN,unknown,source_column,62292.8,2015-01,2015,2015-01-01,daily_household_transactions,0,3,Thursday,1,2015-01-01,0,1,0.0,1.0,1.0,1,late_night,0.5,neutral,0,4.962845,0.333282,0.555547,0.539468,0,daily_household_transactions_unknown_unknown,11,952.0,0.149160,1.640756,0.0,0.638887,watch
2,daily_household_transactions,Daily Household Transactions,household_finance,Daily Household Transactions.csv,NaN,2015-01-01 00:00:00,20.0,20.0,Transportasi,share auto - Place H to Place T base,Expense,unknown,Cash,unknown,unknown,NaN,unknown,source_column,62292.8,2015-01,2015,2015-01-01,daily_household_transactions,0,3,Thursday,1,2015-01-01,0,1,0.0,1.0,1.0,1,late_night,0.5,neutral,0,3.044522,-0.634824,0.394196,0.116564,0,daily_household_transactions_unknown_unknown,11,952.0,0.021008,0.231092,0.0,0.598549,watch
3,daily_household_transactions,Daily Household Transactions,household_finance,Daily Household Transactions.csv,NaN,2015-01-01 00:00:00,40.0,40.0,Hiburan,monument,Expense,unknown,Cash,unknown,unknown,NaN,unknown,source_column,62292.8,2015-01,2015,2015-01-01,daily_household_transactions,0,3,Thursday,1,2015-01-01,0,1,0.0,1.0,1.0,1,late_night,1.0,hedonic,1,3.713572,-0.476118,0.420647,0.297751,0,daily_household_transactions_unknown_unknown,11,952.0,0.042017,0.462185,0.0,0.755162,high
4,daily_household_transactions,Daily Household Transactions,household_finance,Daily Household Transactions.csv,NaN,2015-01-01 00:00:00,200.0,200.0,Makanan,Temple Prasad,Expense,unknown,Cash,unknown,unknown,NaN,unknown,source_column,62292.8,2015-01,2015,2015-01-01,daily_household_transactions,0,3,Thursday,1,2015-01-01,0,1,0.0,1.0,1.0,1,late_night,0.5,neutral,0,5.303305,0.793529,0.632255,0.597955,0,daily_household_transactions_unknown_unknown,11,952.0,0.210084,2.310924,0.0,0.658064,watch


## 11. Feature Summary Helper


In [11]:
def build_feature_summary(frame):
    return (
        frame.groupby('dataset_id')
        .agg(
            rows=('amount', 'size'),
            avg_signal=('fingo_impulse_signal', 'mean'),
            avg_amount_score=('amount_score', 'mean'),
            night_share=('is_night', lambda value: value.mean() * 100),
            hedonic_share=('is_hedonic_category', lambda value: value.mean() * 100),
            high_signal_share=('signal_band', lambda value: (value == 'high').mean() * 100),
        )
        .reset_index()
    )


## 12. Feature Summary


In [12]:
feature_summary_df = build_feature_summary(df_features)
display(feature_summary_df)


,dataset_id,rows,avg_signal,avg_amount_score,night_share,hedonic_share,high_signal_share
0,daily_household_transactions,2445,0.604584,0.641580,61.349693,18.609407,13.987730
1,ecommerce_2024_01_january,367,0.615736,0.588102,28.882834,97.820163,15.531335
2,ecommerce_2024_06_june,608,0.618909,0.598471,23.848684,97.697368,17.434211
3,ecommerce_2025_02_february,831,0.612914,0.613049,25.752106,87.845969,14.079422
4,ecommerce_2025_11_november,1022,0.626811,0.611172,23.972603,98.825832,16.438356


## 13. Visualisasi Feature Helper


In [13]:
def build_feature_metric_frame(summary):
    return summary.melt(
        id_vars='dataset_id',
        value_vars=['avg_signal', 'night_share', 'hedonic_share', 'high_signal_share'],
        var_name='metric',
        value_name='value',
    )


def build_hour_summary(frame):
    return frame.groupby('hour').agg(rows=('amount', 'size'), avg_signal=('fingo_impulse_signal', 'mean')).reset_index()


def build_category_signal_summary(frame):
    return frame.groupby('category').agg(rows=('amount', 'size'), avg_signal=('fingo_impulse_signal', 'mean')).reset_index()


def build_signal_band_summary(frame):
    band_order = ['low', 'medium', 'watch', 'high']
    band = frame['signal_band'].value_counts().reindex(band_order, fill_value=0).reset_index()
    band.columns = ['signal_band', 'rows']
    return band


def plot_feature_overview(summary, frame):
    metric_chart = alt.Chart(build_feature_metric_frame(summary)).mark_bar().encode(
        x=alt.X('value:Q', title='Nilai'),
        y=alt.Y('dataset_id:N', sort='-x', title='Dataset'),
        color=alt.Color('metric:N', title='Metric'),
        row=alt.Row('metric:N', title=None),
        tooltip=['dataset_id', 'metric', alt.Tooltip('value:Q', format='.2f')],
    ).properties(title='Ringkasan Feature per Dataset', height=130)

    hour_chart = alt.Chart(build_hour_summary(frame)).mark_line(point=True).encode(
        x=alt.X('hour:O', title='Jam'),
        y=alt.Y('rows:Q', title='Jumlah Transaksi'),
        tooltip=['hour', 'rows', alt.Tooltip('avg_signal:Q', format='.3f')],
    ).interactive().properties(title='Pola Transaksi per Jam', height=260)

    category_chart = alt.Chart(build_category_signal_summary(frame)).mark_bar().encode(
        x=alt.X('avg_signal:Q', title='Average Signal'),
        y=alt.Y('category:N', sort='-x', title='Kategori'),
        color=alt.Color('category:N', legend=None),
        tooltip=['category', 'rows', alt.Tooltip('avg_signal:Q', format='.3f')],
    ).properties(title='Average Signal per Kategori', height=260)

    band_chart = alt.Chart(build_signal_band_summary(frame)).mark_bar().encode(
        x=alt.X('signal_band:N', sort=['low', 'medium', 'watch', 'high'], title='Signal Band'),
        y=alt.Y('rows:Q', title='Jumlah Transaksi'),
        color=alt.Color('signal_band:N', legend=None),
        tooltip=['signal_band', 'rows'],
    ).properties(title='Distribusi Signal Band', height=260)

    return metric_chart & ((hour_chart | category_chart) & band_chart)


## 14. Visualisasi Feature (Interactive Altair)


In [14]:
plot_feature_overview(feature_summary_df, df_features)


alt.VConcatChart(...)

## 15. Output Feature Engineering


In [15]:
print('Data siap dipakai: df_features, feature_summary_df, new_feature_columns')


Data siap dipakai: df_features, feature_summary_df, new_feature_columns


## Area Analisis Mandiri
Gunakan cell kosong di bawah untuk eksplorasi fitur setelah semua cell utama dijalankan.

Function dan variabel yang bisa dipakai ulang:
- `load_cleaned_from_csv(folder)`: membaca semua file `*_cleaned.csv` dari folder cleaned.
- `prepare_cleaned_input(frame)`: memastikan kolom penting bersih, timestamp valid, amount numerik, dan kategori standar.
- `resolve_cleaned_input(df_cleaned)`: memilih sumber data dari memory notebook 03 atau fallback CSV.
- `run_feature_engineering(frame)`: menjalankan seluruh pipeline fitur waktu, kategori, amount, context, dan signal.
- `get_new_feature_columns(df_input, df_features)`: melihat kolom fitur baru yang ditambahkan.
- `build_feature_summary(df_features)`: membuat ringkasan fitur per dataset.
- `plot_feature_overview(feature_summary_df, df_features)`: membuat visualisasi interaktif hasil feature engineering.
- `df_features`: dataframe utama dengan fitur baru.
- `feature_summary_df`: ringkasan fitur per dataset.
